In [15]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Load the tokenizer and model
model_name = "microsoft/deberta-large-mnli"  # DeBERTa model fine-tuned on the MultiNLI dataset
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Define the premise and generation
ground_truth = " canon"
generation = "'One sponsor of this event is HSBC. The image shows a woman playing tennis on a court, and there is a HSBC logo in the background. This suggests that HSBC is supporting and sponsoring the tennis event, possibly as part of their marketing and branding efforts."

# Tokenize the inputs
inputs = tokenizer(generation, ground_truth, return_tensors='pt', truncation=True)

# Perform inference
with torch.no_grad():
    outputs = model(**inputs)

# Get the predicted label
logits = outputs.logits
predicted_class_id = torch.argmax(logits, dim=1).item()

# Map the predicted class ID to the entailment label
# labels = ["entailment", "neutral", "contradiction"]
labels = ["contradiction", "neutral", "entailment"]
# labels 
predicted_label = labels[predicted_class_id]

print(f"ground_truth: {ground_truth}")
print(f"generation: {generation}")
print(f"Predicted label: {predicted_label}")

Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


ground_truth:  canon
generation: 'One sponsor of this event is HSBC. The image shows a woman playing tennis on a court, and there is a HSBC logo in the background. This suggests that HSBC is supporting and sponsoring the tennis event, possibly as part of their marketing and branding efforts.
Predicted label: neutral


In [5]:
premise = "What is the one sponsor of the event?  canon"

responses = ['One sponsor of this event is HSBC. The image shows a woman playing tennis on a court, and there is a HSBC logo in the background. This suggests that HSBC is supporting and sponsoring the tennis event, possibly as part of their marketing and branding efforts.', 'HSBC is one of the sponsors of this event.', 'In the image, there is a blue wall with a Cathay Pacific logo on it. Cathay Pacific is one of the sponsors of the event, which is a tennis match. The presence of the logo on the wall indicates that the airline company is supporting the event, possibly as a part of their marketing strategy or to promote their brand among the attendees and the audience.', 'In the image, there is a woman playing tennis, and there are two chairs in the background. One chair is closer to the tennis court, while the other chair is located further away. The presence of these chairs suggests that this tennis match might be part of a larger event or competition. One of the sponsors of this event is HSBC, as indicated by the presence of the HSBC logo in the scene.', 'The image shows a woman in a tennis dress, holding a racquet and hitting a tennis ball on a court. The presence of a TV screen in the background suggests that this event is being broadcasted or recorded. One of the sponsors of this tennis event is HP, as indicated by the TV screen. HP is a company that specializes in computer hardware and electronics, and they are likely providing support and equipment for the event, such as the TV screen, cameras, and other technology.', 'One sponsor of this event is HSBC.', 'One sponsor of this event is HSBC, as indicated by the presence of their logo on the backdrop.', 'The image shows a young woman playing tennis on a court, and in the background, there is a sign that reads "Cathay Pacific." This suggests that the tennis match is sponsored by Cathay Pacific, a Hong Kong-based airline.', 'One sponsor of this event is HSBC, which can be seen in the background. HSBC provides financial services and is a prominent sponsor of various athletic events and sports teams. In this image, the tennis player is wearing a gray shirt, which could potentially be an advertisement for the HSBC brand.', 'The event is sponsored by HSBC, as evidenced by the presence of their logo in the image.', 'One sponsor of this event is HSBC. In the image, a woman is playing tennis, and there are multiple chairs visible around the court. One of the chairs is located near the player, and another one is situated further away. The presence of these chairs suggests that there might be spectators or additional staff around the court, indicating that the event is organized and possibly sponsored by HSBC.', 'The image shows a woman playing tennis in a skirt and a grey shirt, holding a tennis racket and swinging it at a ball on a tennis court. She is surrounded by a few chairs. One of the chairs has the word "Hitachi" on it, indicating that Hitachi is one of the sponsors of this tennis event.', 'The image shows a tennis player holding a tennis racket and hitting a tennis ball on a court. The event is sponsored by the company Cathay Pacific, as indicated by the presence of their logo in the background. Cathay Pacific is a Hong Kong-based airline known for its extensive international routes and services. This sponsorship suggests that the tennis event may be a part of a larger sports or entertainment event organized by Cathay Pacific, aiming to promote their brand and attract a broader audience.', 'One sponsor of this event is HP.']

In [12]:
def get_accuracy(full_answers, question, responses, tokenizer, model, device):
    """
    Calculate accuracy by checking entailment of responses with full answers.

    Args:
        full_answers (str): The correct full answer.
        question (str): The associated question.
        responses (list): List of response strings.
        tokenizer: DeBERTa tokenizer.
        model: DeBERTa model.
        device (str): Device for computation.

    Returns:
        float: Accuracy score.
    """
    correct = 0
    for response in responses:
        # premise = f"{question} {full_answers}"
        # hypothesis = f"{question} {response}"
        generatiom = f"{question} {response}"
        ground_truth = f"{question} {full_answers}"
        # generatiom = f'{response}'
        # ground_truth = f'{full_answers}'
        
        inputs = tokenizer.encode_plus(generatiom, ground_truth, return_tensors='pt', truncation=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}  # Move inputs to device
        
        with torch.no_grad():
            logits = model(**inputs).logits
            pred = torch.argmax(logits, dim=1).item()
            if pred == 2:
                correct += 1
    accuracy = correct / len(responses) if responses else 0
    return accuracy


In [13]:
get_accuracy('canon', 'What is the one sponsor of the event?', responses, tokenizer, model, 'cpu')

0.8571428571428571